# Crop Disease Detection — GPU Training on Google Colab (self-contained)

Trains the MobileNetV2 (and optionally ResNet-50) tomato-leaf disease classifier on a free Colab **T4 GPU** in ~1–2 hours (vs. ~37–48 h on CPU). It runs the project's existing `.py` files **unchanged**.

**You upload nothing large.** The dataset is fetched automatically from its public GitHub source *on Colab* (not from your machine). The only thing you provide is the 6 small code files, when **Step 3** prompts you.

**Before you start:**
1. Menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
2. Then **Runtime → Run all** (or run the cells top to bottom).

When Step 3 prompts, select these 6 files from your project folder: `config.py`, `dataset_preparation.py`, `preprocessing.py`, `model.py`, `train.py`, `evaluate.py`.

> Everything runs on Colab's fast local disk (`/content`). No Google Drive is required. If the runtime disconnects mid-train, just re-run from Step 3 (a fresh GPU run is only ~1–2 h). To make it disconnect-proof instead, see the optional Drive note at the end.

## Step 1 — Confirm the GPU runtime is active

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs visible:', gpus)
assert gpus, ('No GPU detected! Runtime > Change runtime type > T4 GPU, '
              'then Runtime > Restart runtime, and run this cell again.')
print('GPU runtime OK.')

## Step 2 — Keras-2 compatibility (only if Colab ships Keras 3)

Your local environment uses TensorFlow 2.12 (Keras 2). If Colab's TensorFlow is 2.16+ it defaults to **Keras 3**, whose saved `.h5` will not load back in your local Keras 2. This installs `tf_keras` and selects legacy (Keras 2) mode in that case, so the model you download loads cleanly in your local `evaluate.py` / `app.py`.

In [ ]:
import sys, subprocess
import tensorflow as tf
from packaging import version

if version.parse(tf.__version__) >= version.parse('2.16'):
    v = tf.__version__
    print(f'TF {v} ships Keras 3 -> installing tf_keras=={v} for Keras-2 compatibility...')
    if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'tf_keras=={v}']).returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tf_keras'])
    USE_LEGACY = '1'
else:
    print(f'TF {tf.__version__} already uses Keras 2 — nothing to install.')
    USE_LEGACY = '0'

print('TF_USE_LEGACY_KERAS =', USE_LEGACY)

## Step 3 — Provide the 6 code files

Run this cell, click **Choose Files**, and select all 6 from your project folder:
`config.py`, `dataset_preparation.py`, `preprocessing.py`, `model.py`, `train.py`, `evaluate.py`.
(They are a few KB each — this is the only upload, and it is tiny.)

In [ ]:
import os
os.chdir('/content')
from google.colab import files
uploaded = files.upload()

required = ['config.py', 'dataset_preparation.py', 'preprocessing.py',
            'model.py', 'train.py', 'evaluate.py']
missing = [f for f in required if not os.path.isfile(f)]
assert not missing, f'Still missing: {missing} — re-run this cell and select them.'
print('All 6 code files present in', os.getcwd())

## Step 4 — Fetch the dataset from its public source (no upload)

Sparse, blob-less clone of just the 6 tomato classes from `spMohanty/PlantVillage-Dataset` (the same source the project was built from), then rename the repo's `Tomato___X` folders to the `Tomato__X` convention used in `config.py`. Downloads on Colab's network (~180 MB, a minute or two), not from your machine. Expect a total of **12,936** images.

In [ ]:
import os, shutil

REPO = 'https://github.com/spMohanty/PlantVillage-Dataset.git'
classes = ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
           'Tomato___Leaf_Mold', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___healthy']
sparse = ' '.join(f'raw/color/{c}' for c in classes)

!rm -rf pv data/PlantVillage
!git clone --filter=blob:none --no-checkout --depth 1 {REPO} pv
!cd pv && git sparse-checkout init --cone && git sparse-checkout set {sparse} && git checkout

os.makedirs('data/PlantVillage', exist_ok=True)
for c in classes:
    dst = c.replace('Tomato___', 'Tomato__')   # triple -> double underscore
    shutil.move(f'pv/raw/color/{c}', f'data/PlantVillage/{dst}')
shutil.rmtree('pv', ignore_errors=True)

total = 0
for d in sorted(os.listdir('data/PlantVillage')):
    n = len(os.listdir(f'data/PlantVillage/{d}'))
    total += n
    print(f'  {d:<45} {n:>5}')
print(f'  TOTAL: {total}  (expected 12,936)')
assert total == 12936, f'Expected 12,936 images, got {total}.'
print('Dataset ready at data/PlantVillage/')

## Step 5 — Regenerate the stratified split

Creates `checkpoints/dataset_split.npz`. Same seed (42) + same images ⇒ the 70/15/15 partition is identical to the one produced locally. Confirm the printed counts read Train 9055 / Val 1940 / Test 1941.

In [ ]:
!TF_USE_LEGACY_KERAS={USE_LEGACY} python dataset_preparation.py

## Step 6 — Train MobileNetV2 (primary model) on the GPU

Full two-phase schedule: 20 epochs head-only + 30 epochs fine-tuning. On a T4 this is roughly **1–2 hours**. If the runtime drops, re-run Steps 3–6 (checkpoints live on Colab's local disk; re-running this cell auto-resumes from the last completed epoch within the same session).

In [ ]:
!TF_USE_LEGACY_KERAS={USE_LEGACY} python train.py --backbone mobilenetv2

## Step 7 — (Optional) Train ResNet-50 (comparative model)

Run only if you also need the comparative ResNet-50 results. Heavier than MobileNetV2.

In [ ]:
!TF_USE_LEGACY_KERAS={USE_LEGACY} python train.py --backbone resnet50

## Step 8 — Evaluate on the held-out test set

Produces `<backbone>_classification_report.txt` and `<backbone>_confusion_matrix.png` in `checkpoints/`.

In [ ]:
!TF_USE_LEGACY_KERAS={USE_LEGACY} python evaluate.py --backbone mobilenetv2

## Step 9 — Download your results

Downloads the trained model + plots + report to your computer. Put `mobilenetv2_best.h5` into your local `checkpoints/` folder to run the Streamlit app locally (`streamlit run app.py`).

In [ ]:
import os
from google.colab import files

print('Contents of checkpoints/:')
for f in sorted(os.listdir('checkpoints')):
    size = os.path.getsize(os.path.join('checkpoints', f)) / 1e6
    print(f'  {f:<42} {size:8.2f} MB')

for f in ['mobilenetv2_best.h5', 'mobilenetv2_training_history.png',
          'mobilenetv2_classification_report.txt', 'mobilenetv2_confusion_matrix.png']:
    p = os.path.join('checkpoints', f)
    if os.path.isfile(p):
        files.download(p)

---
### Optional: make it disconnect-proof with Google Drive

The flow above keeps everything on Colab's temporary disk, so a disconnect means re-running (cheap at ~1–2 h on GPU). If you'd rather have checkpoints persist and resume across disconnects, run this **before Step 5**, then the checkpoints survive and `train.py` resumes from your Drive:

```python
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/crop_disease_ckpts', exist_ok=True)
# symlink so the scripts' relative checkpoints/ path points at Drive:
if not os.path.islink('checkpoints'):
    os.symlink('/content/drive/MyDrive/crop_disease_ckpts', 'checkpoints')
```

(Mounting Drive shows a Google sign-in / consent popup — that step needs you.)